# Egitim notebook'u: `qwen3vl_8b`

- Model: `Qwen/Qwen3-VL-8B-Instruct` (family: `qwen`)
- Ortam: Colab A100 80 GB, bf16, TF32 acik.
- Tahmini sure: ~3-6 saat (veri boyutuna ve epoch sayisina gore).
- Ciktilar: Drive'da `kap_lora/qwen3vl_8b/` altina yazilir (dogrudan Drive'a).
- Sonunda `merge_lora.py` ile adapter birlestirilir.


In [ ]:
# Ayarlar
MODEL_KEY = "qwen3vl_8b"
BACKEND = "unsloth"
LOAD_IN_4BIT = False
BATCH = 8
GRAD_ACCUM = 4
EPOCHS = 3
LR = 2e-4
LORA_R = 32
DRIVE_DIR = "/content/drive/MyDrive/kap_lora"
DATASET_ZIP = f"{DRIVE_DIR}/dataset.zip"
REPO_URL = "github.com/furkancmc/finance-vl-fine-tune"


In [ ]:
# Kurulum: Drive, paketler, HF token
from google.colab import drive
drive.mount("/content/drive")
!pip install -q unsloth bitsandbytes peft accelerate qwen-vl-utils rapidfuzz
import os
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")
    if tok:
        from huggingface_hub import login
        login(tok)
        print("HF login ok")
except Exception as e:
    print("HF token yok/atlandi:", e)


In [ ]:
# Repo cek: GH_TOKEN varsa private clone, yoksa Drive'dan kopyala
import os, shutil, subprocess
from pathlib import Path
gh = None
try:
    from google.colab import userdata
    gh = userdata.get("GH_TOKEN")
except Exception:
    pass
if os.path.exists("/content/repo"):
    shutil.rmtree("/content/repo")
if gh:
    url = f"https://{gh}@{REPO_URL}.git"
    subprocess.run(["git", "clone", url, "/content/repo"], check=True)
else:
    src = Path(DRIVE_DIR) / "code"
    assert src.exists(), f"GH_TOKEN yok ve {src} bulunamadi"
    shutil.copytree(src, "/content/repo")
%cd /content/repo


In [ ]:
# Veri: dataset.zip ac ve train.jsonl satir sayisi
import os, zipfile
from pathlib import Path
if not os.path.exists("/content/dataset"):
    assert os.path.exists(DATASET_ZIP), f"zip yok: {DATASET_ZIP}"
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall("/content")
p = Path("/content/dataset/train.jsonl")
n = sum(1 for _ in p.open("r", encoding="utf-8"))
print("train satir:", n)


In [ ]:
# GPU kontrol
!nvidia-smi
import torch
print(torch.cuda.get_device_name(0))
print("bellek GB:", torch.cuda.get_device_properties(0).total_memory / 1e9)
import transformers
from packaging.version import Version
from model_zoo import get
need = get(MODEL_KEY)["min_transformers"]
print("transformers", transformers.__version__, "gerekli >=", need)
assert Version(transformers.__version__) >= Version(need), "transformers cok eski: !pip install -U transformers"


## Smoke test (kisa, hizli dogrulama)

In [ ]:
!python train_lora.py --model $MODEL_KEY --backend $BACKEND \
  --limit 32 --max-steps 3 --batch 4 --grad-accum 1 \
  --out /content/smoke --eval-steps 3 --save-steps 3 --max-eval 8


## Tam egitim (Drive'a yaz, --resume ile devam edilebilir)

In [ ]:
FLAG_4BIT = "--load-in-4bit" if LOAD_IN_4BIT else ""
!python train_lora.py --model $MODEL_KEY --backend $BACKEND \
  --data /content/dataset \
  --batch $BATCH --grad-accum $GRAD_ACCUM --epochs $EPOCHS --lr $LR \
  --lora-r $LORA_R $FLAG_4BIT \
  --out {DRIVE_DIR}/qwen3vl_8b --resume


## Kayip egrisi

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
log = Path(f"{DRIVE_DIR}/qwen3vl_8b/train_log.jsonl")
steps, losses, ev_steps, ev_losses = [], [], [], []
for line in log.open("r", encoding="utf-8"):
    r = json.loads(line)
    if "loss" in r and r.get("step") is not None:
        steps.append(r["step"]); losses.append(r["loss"])
    if "eval_loss" in r and r.get("step") is not None:
        ev_steps.append(r["step"]); ev_losses.append(r["eval_loss"])
plt.figure(figsize=(8,4))
plt.plot(steps, losses, label="train")
if ev_steps:
    plt.plot(ev_steps, ev_losses, label="eval")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(True)
plt.show()


## LoRA birlestirme

In [ ]:
!python merge_lora.py --model $MODEL_KEY \
  --adapter {DRIVE_DIR}/qwen3vl_8b/final \
  --out {DRIVE_DIR}/qwen3vl_8b/merged


## Sonraki adim

Degerlendirme icin **yeni bir runtime** acin ve `eval_qwen3vl_8b.ipynb` notebook'unu calistirin (vLLM unsloth ile ayni ortama kurulmaz).
